# 23. Conversion rates when the denominator is zero

**Track:** sql | **Difficulty:** easy | **Tags:** division by zero, nullif, rates, nulls

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("023-conversion-rate-zero-denominator")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "campaign_stats": """
UEFSMRUEFcAHFfoDTBV4FQASAADgAwjLDwAFAQDODQgAtA0IAL0NCADcDQgAuQ0IALcNCADWDQgA1A0IALINCACiDQgAyQ0IAMQN
CAC8DQgAvg0IAKcNCADKDQgA2g0IALMNCAC4DQgAoQ0IAMgNCADYDQgArw0IAKoNCADADQgAtg0IANENCACtDQgAqQ0IAM0NCADD
DQgArg0IALENCAC1DQgA2Q0IANANCAC7DQgA1Q0IAKMNCADSDQgAxg0IAM8NCADFDQgAqA0IALoNCACwDQgAzA0IANMNCADHDQgA
pQ0IAKwNCACmDQgA2w0IAKQNCACrDQgAwg0IANcNCDzBDwAAAAAAAL8PAAAAAAAAFQAVcBV0LBV4FRAVBhUGHBgI3A8AAAAAAAAY
CKEPAAAAAAAAFgAoCNwPAAAAAAAAGAihDwAAAAAAABERAAAAONwCAAAAeAEGEUAgDERhHEiiLEzjPFAkTVRlXVimbVznfWAojmRp
nmiqrmzrvnAsz3Rt33iu7wAAABUEFVAVVEwVCBUAEgAAKJwGAAAAc2VhcmNoBwAAAGRpc3BsYXkFAAAAZW1haWwGAAAAc29jaWFs
FQAVMBU0LBV4FRAVBhUGHDYAKAZzb2NpYWwYB2Rpc3BsYXkREQAAABhcAgAAAHgBAhEQJozqRY/dOObxPfz8tCIAFQQV8AYVsARM
FW4VABIAALgDAAANAQQ1EwkJBJY8CQgElI8JCATYRQkIBJ9eCQgEhyYJCAT0bQkIBHgRCQgED0cJCAQYOwkIBAOMCQgEeLAJCATT
KgkIBBhoCQgI/mkABQEENrEFBwgAMMEJCASlpQkIAM4NeATnDgkQBJcrCQgEEw0JCATPugkIBGI4CQgEKUAJCAQJTAkIBDZjCQgA
Tw04AEMNEASxigkYBH5GCQgE1BkJCAT/iwkIBJSiCQgEm0EJCAQemQkIBBkaCQgE4RcJCAStUAkIBBI5CQgAOQ1IBJ61CRAEgFgJ
CATSIgkIBEG5CQgEwZUJCATzdAkIBPYQCQgEgz0JCAQgFgkIAIYtCARlcwkQPOuDAAAAAAAAgh4AAAAAAAAVABVwFXQsFXgVEBUG
FQYcGAgwwQAAAAAAABgIAAAAAAAAAAAWACgIMMEAAAAAAAAYCAAAAAAAAAAAEREAAAA43AIAAAB4AQYRQCAMRGEcSKIsAMA0zgNF
0kRV1oVl2gZwnfeBoTiSpQmcaKquANu6LxzLM13bAAAAFQQV4AUVogNMFVwVABIAAPACAAANAQQJBAkJBPwJCQgEMwUJCAQHAgkI
APUJBwQAEA0QAJ0NEACgDSgA6Q0QBKEICQkElwwJCAB0DTAASg0YAGMNaABvDWgANg0gBBwDCTAAzQkHCABtAQkJAA0NGAAYDTgA
Nw0YAEcNUACzCScEAOwNGAAWDVAAww0IAEUNGACcDagAUQ0wANoNGABhDVgAjg0QBKoNCSEAlQkHBADlDRgA9A2wAOgNWADADVAA
Sw0oAAYNKAQhBwkRAJENCDywCQAAAAAAANIAAAAAAAAAFQAVdBV4LBV4FRAVBhUGHBgIqg0AAAAAAAAYCAAAAAAAAAAAFgwoCKoN
AAAAAAAAGAgAAAAAAAAAABERAAAAOuQKAAAABf7vHgEJ/Pt/HwYPQCAMRGEcSKIAC9M4wANF0gQAVAVYwIVl2sZ1HvCBoTgCZGme
aKqubAsAFQQVsAUV+AJMFVYVABIAANgCAAANAQCoDQgAHw0IANMNCABnDQgADQ0IADMNCAALDQgAvQ0IAAYNCAQ3AQkJAOwNCAAu
CQ8EABsNEAC0DQgAVQ0YAGMNCAAXDQgAEg0IAEINCACrDQgANQ0IAE8NCABWDQgAVA0IAGINCAAkDQgAFg0IAAcNCAA8DQgAGg0I
AFINCABFDQgE0QEJCQARCQcEAFwNCAAhDRgA8Q0QAAoNCAArDQgEVwAJATyyAQAAAAAAAAgAAAAAAAAAFQAVeBV8LBV4FRAVBhUG
HBgI7AEAAAAAAAAYCAAAAAAAAAAAFg4oCOwBAAAAAAAAGAgAAAAAAAAAABERAAAAPOwMAAAAB/7v/RQBA/oWAQNuBg9AIAxEYRxI
ogAL0wDOA0USAEwUUAHWc2GZtnEd4H1gKAIjKZbmiaYqAAAVBBlsNQAYBnNjaGVtYRUKABUEJQIYC2NhbXBhaWduX2lkABUMJQIY
B2NoYW5uZWwlAEwcAAAAFQQlAhgLaW1wcmVzc2lvbnMAFQQlAhgGY2xpY2tzABUEJQIYB3NpZ251cHMAFngZHBlcJgAcFQQZNQAG
EBkYC2NhbXBhaWduX2lkFQIWeBbOCRaMBiaiBCYIHBgI3A8AAAAAAAAYCKEPAAAAAAAAFgAoCNwPAAAAAAAAGAihDwAAAAAAABER
ABksFQQVABUCABUAFRAVAgA8KQYZJgB4AAAAJgAcFQwZNQAGEBkYB2NoYW5uZWwVAhZ4FuwBFvQBJoQHJpQGHDYAKAZzb2NpYWwY
B2Rpc3BsYXkREQAZLBUEFQAVAgAVABUQFQIAPBbMBRkGGSYAeAAAACYAHBUEGTUABhAZGAtpbXByZXNzaW9ucxUCFngW/ggWwgYm
2AwmiAgcGAgwwQAAAAAAABgIAAAAAAAAAAAWACgIMMEAAAAAAAAYCAAAAAAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAHgA
AAAmABwVBBk1AAYQGRgGY2xpY2tzFQIWeBbyBxa4BSaMEibKDhwYCKoNAAAAAAAAGAgAAAAAAAAAABYMKAiqDQAAAAAAABgIAAAA
AAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYMbAAAACYAHBUEGTUABhAZGAdzaWdudXBzFQIWeBbGBxaSBSaaFyaCFBwYCOwB
AAAAAAAAGAgAAAAAAAAAABYOKAjsAQAAAAAAABgIAAAAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYOagAAABbwIxZ4JggW
jBkAGRwYDEFSUk9XOnNjaGVtYRjMAy8vLy8vMUFCQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlB
QUFBQkFBSUFBQUFCQUFBQUFVQUFBRGtBQUFBcEFBQUFHd0FBQUE0QUFBQUJBQUFBRUQvLy84QUFBRUNFQUFBQUJnQUFBQUVBQUFB
QUFBQUFBY0FBQUJ6YVdkdWRYQnpBQ3ovLy84QUFBQUJRQUFBQUhELy8vOEFBQUVDRUFBQUFCZ0FBQUFFQUFBQUFBQUFBQVlBQUFC
amJHbGphM01BQUZ6Ly8vOEFBQUFCUUFBQUFLRC8vLzhBQUFFQ0VBQUFBQndBQUFBRUFBQUFBQUFBQUFzQUFBQnBiWEJ5WlhOemFX
OXVjd0NRLy8vL0FBQUFBVUFBQUFEVS8vLy9BQUFCQlJBQUFBQWNBQUFBQkFBQUFBQUFBQUFIQUFBQVkyaGhibTVsYkFBRUFBUUFC
QUFBQUJBQUZBQUlBQVlBQndBTUFBQUFFQUFRQUFBQUFBQUJBaEFBQUFBa0FBQUFCQUFBQUFBQUFBQUxBQUFBWTJGdGNHRnBaMjVm
YVdRQUNBQU1BQWdBQndBSUFBQUFBQUFBQVVBQUFBQT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRlcHAAAHAAA
HAAAHAAAHAAAAJcEAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "023-conversion-rate-zero-denominator"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

campaign_stats = spark.read.parquet((_data_dir / "campaign_stats.parquet").resolve().as_posix())
campaign_stats.createOrReplaceTempView("campaign_stats")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'campaign_id': '0b83c74081573d8f13f859ae04f6c07e9cb71b914d749e2852a0563bb8c3b94a',
                         'click_rate': 'b4cca8657abc6178d8874698394d61b06feaca40b84402b12a7fd65fe0a1ec54',
                         'signup_rate': '0eb353d8350d28303d16c22c4393451504c41a780a6e267f5cb41902beba215e'},
 'columns': [['campaign_id', 'bigint'], ['click_rate', 'double'], ['signup_rate', 'double']],
 'fingerprint': '9b27efd0b765813ad2e0fed754f023eb261aabebe09d699e0ae0770237be89f3',
 'order_matters': False,
 'row_count': 60}

check = make_check(EXPECTED)

## The situation

Marketing ranks campaigns by two rates: how often an ad is clicked, and how often a
click turns into a signup.

Some campaigns were never shown, some were shown and never clicked, and for a few
the click or signup tracking was broken, so the count is missing. The current report
either stops with a division error or shows those campaigns with a rate of 0, which
puts a campaign that was never shown at the bottom of the ranking next to the truly
bad ones.

## Input tables

### `campaign_stats`

One row per campaign.

| Column | Type | Description |
|---|---|---|
| `campaign_id` | BIGINT | Unique per campaign. |
| `channel` | STRING | `search`, `social`, `email` or `display`. |
| `impressions` | BIGINT | How many times the ad was shown. Never NULL, can be 0. |
| `clicks` | BIGINT | How many times it was clicked. NULL if click tracking was broken. |
| `signups` | BIGINT | How many signups followed. NULL if signup tracking was broken. |

## Requirement

Return both rates for every campaign.

- `click_rate` is `clicks` divided by `impressions`.
- `signup_rate` is `signups` divided by `clicks`.
- A rate is NULL when it cannot be computed: the number it is divided by is 0 or
  NULL, or the number being divided is NULL.
- A rate of 0 means something specific: the ad was shown (or clicked) and nothing
  followed. A campaign with 2,000 impressions and 0 clicks has `click_rate` 0 and
  `signup_rate` NULL.
- Do not round.

## Expected output

One row per campaign.

| Column | Type | Description |
|---|---|---|
| `campaign_id` | BIGINT | |
| `click_rate` | DOUBLE | NULL if it cannot be computed. |
| `signup_rate` | DOUBLE | NULL if it cannot be computed. |

In [ ]:
campaign_stats.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `campaign_stats`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)